In [ ]:
#!/usr/bin/env python3
"""
YOLOP Lane Segmentation Script

This script uses the pretrained YOLOP model to perform lane line segmentation on a batch 
of video frame images. Given an input directory of frames (extracted from videos), it runs 
YOLOP inference on each image and saves an annotated copy with lane markings overlaid.

Usage:
    python yolop_lane_seg.py --input <frames_dir> --output <output_dir> [--device cpu/cuda]

Requirements:
    - Clone the YOLOP repository and install requirements (PyTorch 1.7+, etc.):contentReference[oaicite:5]{index=5}.
    - Ensure the YOLOP model weights are available. By default this script uses PyTorch Hub 
      to load the pretrained YOLOP model automatically:contentReference[oaicite:6]{index=6}.
    - Install OpenCV (cv2) for image processing.

The output images will be saved in the specified output directory, preserving the subfolder 
structure and filenames of the input frames directory.

Note: Lane lines are overlaid in red with partial transparency. This script focuses on lane 
segmentation; YOLOP's object detection and drivable area outputs are not drawn for clarity.
"""
import os
import cv2
import torch
import argparse
import numpy as np
from pathlib import Path

def letterbox_image(img, new_size=(640, 640), color=(114, 114, 114)):
    """
    Resize an image to fit into a target size while preserving aspect ratio and padding.
    Returns the resized image, the scaling ratio, and the padding (pad_x, pad_y).
    """
    orig_h, orig_w = img.shape[:2]
    target_h, target_w = new_size
    # Calculate scale ratio for resizing, and compute padding
    scale = min(target_w / orig_w, target_h / orig_h)
    new_w, new_h = int(orig_w * scale), int(orig_h * scale)
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    # Compute padding to reach target size
    pad_w = target_w - new_w
    pad_h = target_h - new_h
    # Divide padding equally left-right, top-bottom
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    # Add border padding
    img_padded = cv2.copyMakeBorder(img_resized, pad_top, pad_bottom, pad_left, pad_right,
                                    borderType=cv2.BORDER_CONSTANT, value=color)
    return img_padded, scale, (pad_left, pad_top)

def run_lane_segmentation(input_dir, output_dir, device='cpu'):
    """
    Run lane segmentation on all images in input_dir and save annotated results to output_dir.
    """
    # Load YOLOP model (pretrained on BDD100K) using PyTorch Hub:contentReference[oaicite:7]{index=7}
    model = torch.hub.load('hustvl/yolop', 'yolop', pretrained=True, source='github')
    model.to(device)
    model.eval()  # set to evaluation mode
    
    # Optionally, if using GPU, switch model to half precision for speed
    # (commented out by default for compatibility)
    # if 'cuda' in device:
    #     model.half()
    
    # Walk through all image files in input directory
    input_path = Path(input_dir)
    for img_path in input_path.rglob("*"):
        if img_path.is_dir():
            continue  # skip directories
        # Only process common image file extensions
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp"]:
            continue
        
        # Read image
        orig_img = cv2.imread(str(img_path))
        if orig_img is None:
            print(f"[warn] Skipping unreadable file: {img_path}")
            continue
        
        orig_h, orig_w = orig_img.shape[:2]
        # Prepare image with letterboxing for YOLOP input
        img, scale, (pad_x, pad_y) = letterbox_image(orig_img, new_size=(640, 640))
        # Convert to torch tensor
        # --- FIX: make image contiguous before converting to tensor ---
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # convert BGR→RGB safely
        img_chw = np.transpose(img_rgb, (2, 0, 1)).copy()     # HWC→CHW and make a copy
        img_tensor = torch.from_numpy(img_chw).float().to(device)
        img_tensor /= 255.0                                   # normalize to [0,1]

        if img_tensor.dim() == 3:
            img_tensor = img_tensor.unsqueeze(0)  # add batch dimension
        
        # If using half precision on GPU, uncomment the next line
        # if img_tensor.device.type != 'cpu':
        #     img_tensor = img_tensor.half()
        
        # Run inference (YOLOP outputs: det_out, da_seg_out, ll_seg_out)
        with torch.no_grad():
            det_out, da_seg_out, ll_seg_out = model(img_tensor)
        # YOLOP returns det_out as a tuple (prediction, training_out), ignore detection
        # Extract lane line segmentation output
        # Segmentation outputs might have shape [1,2,H,W] (for two classes: background vs lane)
        # Remove padding from segmentation prediction:
        pad_x, pad_y = int(pad_x), int(pad_y)
        seg_pred = ll_seg_out  # ll_seg_out has lane segmentation logits
        if seg_pred.shape[2] != 0:  # proceed if output is not empty
            seg_pred = seg_pred[:, :, pad_y: seg_pred.shape[2] - pad_y, pad_x: seg_pred.shape[3] - pad_x]
        # Upsample segmentation mask back to original image size
        seg_mask = torch.nn.functional.interpolate(seg_pred, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
        # Convert to binary mask (class 1 = lane, class 0 = background)
        seg_mask = seg_mask.argmax(dim=1).squeeze().cpu().numpy().astype(np.uint8)
        # Overlay lane mask on the original image
        lane_color = (0, 0, 255)  # red in BGR
        overlay = orig_img.copy()
        overlay[seg_mask == 1] = lane_color  # color the lane pixels red
        # Blend the overlay with original image for transparency
        annotated_img = cv2.addWeighted(orig_img, 0.7, overlay, 0.3, 0)
        
        # Construct output path and save the image
        rel_path = img_path.relative_to(input_path)
        out_path = Path(output_dir) / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_path), annotated_img)
        print(f"[info] Processed {img_path} -> {out_path}")

In [ ]:
if __name__ == "__main__":
    input_dir = r"C:\Users\HP\Documents\base\frames\20250826_34402PMByGPSMapCamera"
    output_dir = r"C:\Users\HP\Documents\base\frames\20250826_34402_Result"
    device = "cpu"  # or "cuda" if you have GPU support
    run_lane_segmentation(input_dir, output_dir, device)